<a href="https://colab.research.google.com/github/stekkos89thelawnmower/FLAMINGO-Cosmic-Web-Classification/blob/main/FLAMINGO_QD_cosmic_web_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FLAMINGO Q_D — Cosmic-Web Environment Classification

Reproducible pipeline: kinematical backreaction Q_D (Buchert 2000) computed from the IDW-reconstructed velocity field of the FLAMINGO simulation, extended with a cosmic-web environment classification (V-web, calibrated via percolation analysis, cross-validated against an independent density-based T-web classifier), an exact variance decomposition of Q_D by environment, and a differential fiducial/NoCooling test by environment across three epochs.

Companion technical report: *Cosmic-Web Environment Classification for Kinematical Backreaction (Q_D) Estimation in FLAMINGO*, S. Boi (2026).

## 0. Setup

In [ ]:
!pip install hdfstream -q
import numpy as np
import hdfstream
import time
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import wilcoxon
from scipy import ndimage


In [ ]:
BOX_SIDE = 1000.0
GRAMS_PER_MSUN = 1.988409870698051e33
CM_PER_KM = 1e5
MASS_CUT = 1e12
N_GRID = 200
CELL_SIZE = BOX_SIDE / N_GRID
K_NEIGHBORS = 16
GRID_1D = np.linspace(0, BOX_SIDE, N_GRID, endpoint=False) + CELL_SIZE / 2
SEED = 42
DOMAIN_RADIUS_MPC = 100.0
N_OBSERVERS = 80
MIN_SEPARATION_MPC = 2 * DOMAIN_RADIUS_MPC
WEB_THRESHOLD = 6.7   # calibrated via percolation analysis, Section 3


In [ ]:
def load_tracers_and_velocities(flamingo_dir, run_path, snapshot_file, expected_z=None):
    """Load halo (M > MASS_CUT) positions and peculiar velocities for one
    FLAMINGO run/snapshot."""
    soap_file = flamingo_dir[f"{run_path}/SOAP-HBT/{snapshot_file}"]
    z = float(np.array(soap_file["Header"].attrs["Redshift"]).squeeze())
    if expected_z is not None:
        assert abs(z - expected_z) < 1e-6, f"unexpected z: {z} (expected {expected_z})"
    total_mass_raw = np.array(soap_file["SO/200_crit/TotalMass"][:], dtype=np.float64)
    conv_mass = dict(soap_file["SO/200_crit/TotalMass"].attrs)
    cgs_mass = float(np.array(conv_mass["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    mass_msun = total_mass_raw * cgs_mass / GRAMS_PER_MSUN
    pos_raw = np.mod(np.array(soap_file["SO/200_crit/CentreOfMass"][:]), BOX_SIDE)
    vel_raw = np.array(soap_file["SO/200_crit/CentreOfMassVelocity"][:], dtype=np.float64)
    conv_vel = dict(soap_file["SO/200_crit/CentreOfMassVelocity"].attrs)
    vel_cgs_factor = float(np.array(conv_vel["Conversion factor to physical CGS (including cosmological corrections)"]).squeeze())
    vel_kms = vel_raw * vel_cgs_factor / CM_PER_KM
    mask = mass_msun > MASS_CUT
    return pos_raw[mask], vel_kms[mask], z


def periodic_delta(a, b, box_side):
    d = np.abs(a - b)
    return np.minimum(d, box_side - d)


def make_observer_positions(n_observers, min_separation_mpc, box_side, seed=SEED, max_tries_factor=500):
    """N random, non-overlapping observer positions (min separation = 2 x
    domain radius by default), shared seed across variants -> paired design."""
    rng = np.random.default_rng(seed)
    positions = []
    tries = 0
    max_tries = max_tries_factor * n_observers
    while len(positions) < n_observers and tries < max_tries:
        cand = rng.uniform(0, box_side, size=3)
        if all(np.sqrt((periodic_delta(cand, p, box_side) ** 2).sum()) >= min_separation_mpc for p in positions):
            positions.append(cand)
        tries += 1
    return np.array(positions)


def build_grid_tree(grid_1d, box_side):
    gx, gy, gz = np.meshgrid(grid_1d, grid_1d, grid_1d, indexing="ij")
    grid_points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    return cKDTree(grid_points, boxsize=box_side)


def subsample_to_match(tracers, velocities, target_n, seed):
    """Random subsampling without replacement, to equalise tracer counts
    between variants before field reconstruction."""
    n = len(tracers)
    if n <= target_n:
        return tracers, velocities
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=target_n, replace=False)
    return tracers[idx], velocities[idx]


In [ ]:
VARIANT_A = {"run_path": "L1_m9/L1_m9", "label": "fiducial"}
VARIANT_B = {"run_path": "L1_m9/NoCooling", "label": "NoCooling"}

EPOCHS = [
    ("halo_properties_0057.hdf5", 1.00),
    ("halo_properties_0067.hdf5", 0.50),
    ("halo_properties_0077.hdf5", 0.00),
]

root_dir = hdfstream.open("cosma", "/")
flamingo_dir = root_dir["FLAMINGO"]

grid_tree = build_grid_tree(GRID_1D, BOX_SIDE)
observer_positions = make_observer_positions(N_OBSERVERS, MIN_SEPARATION_MPC, BOX_SIDE, seed=SEED)
print(f"Observer positions: {len(observer_positions)}")


## 1. Baseline Q_D pipeline (IDW velocity field)

The peculiar velocity field is reconstructed by inverse-distance-weighting (IDW) interpolation onto the fixed 200³ grid, then differentiated by centred finite differences. This engine is unchanged throughout this notebook and is the sole source of θ, σ², and the velocity-shear eigenvalues used below. Q_D follows the Buchert (2000) definition with the shear-term coefficient validated in prior work; the vorticity-inclusive general form (Kazimierczak, 2016) is not required here since it is not computed by this grid-based engine.

In [ ]:
def build_velocity_field(tracers, velocities, grid_1d, box_side, n_grid, k_neighbors):
    """IDW (k nearest neighbours) interpolation of tracer velocities onto a
    fixed grid, shared across variants -> paired comparison."""
    tree = cKDTree(tracers, boxsize=box_side)
    vx = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    vy = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    vz = np.empty((n_grid, n_grid, n_grid), dtype=np.float32)
    for i in range(n_grid):
        gx = np.full(n_grid * n_grid, grid_1d[i])
        gyv, gzv = np.meshgrid(grid_1d, grid_1d, indexing="ij")
        pts = np.stack([gx, gyv.ravel(), gzv.ravel()], axis=1)
        d, idx = tree.query(pts, k=k_neighbors)
        d = np.maximum(d, 1e-6)
        w = 1.0 / d**2
        w /= w.sum(axis=1, keepdims=True)
        vx[i] = (w * velocities[idx, 0]).sum(axis=1).reshape(n_grid, n_grid)
        vy[i] = (w * velocities[idx, 1]).sum(axis=1).reshape(n_grid, n_grid)
        vz[i] = (w * velocities[idx, 2]).sum(axis=1).reshape(n_grid, n_grid)
    return vx, vy, vz


def compute_theta_sigma2_eigvals_grid(vx, vy, vz, cell_size):
    """Derive theta, sigma2 (Buchert Q_D shear term) and the velocity-shear
    eigenvalues (V-web classification, Hoffman et al. 2012 sign convention:
    positive = converging flow) from the same velocity-gradient tensor."""
    dvx_dx = (np.roll(vx,-1,0)-np.roll(vx,1,0))/(2*cell_size)
    dvx_dy = (np.roll(vx,-1,1)-np.roll(vx,1,1))/(2*cell_size)
    dvx_dz = (np.roll(vx,-1,2)-np.roll(vx,1,2))/(2*cell_size)
    dvy_dx = (np.roll(vy,-1,0)-np.roll(vy,1,0))/(2*cell_size)
    dvy_dy = (np.roll(vy,-1,1)-np.roll(vy,1,1))/(2*cell_size)
    dvy_dz = (np.roll(vy,-1,2)-np.roll(vy,1,2))/(2*cell_size)
    dvz_dx = (np.roll(vz,-1,0)-np.roll(vz,1,0))/(2*cell_size)
    dvz_dy = (np.roll(vz,-1,1)-np.roll(vz,1,1))/(2*cell_size)
    dvz_dz = (np.roll(vz,-1,2)-np.roll(vz,1,2))/(2*cell_size)
    theta = dvx_dx+dvy_dy+dvz_dz
    grad = np.stack([[dvx_dx,dvx_dy,dvx_dz],[dvy_dx,dvy_dy,dvy_dz],[dvz_dx,dvz_dy,dvz_dz]])
    grad_last = np.moveaxis(grad, [0,1], [-2,-1])
    S_full = 0.5*(grad_last + np.swapaxes(grad_last,-1,-2))
    eigvals = np.linalg.eigvalsh(-S_full)

    sym = 0.5*(grad + grad.transpose(1,0,2,3,4))
    trace_third = theta/3.0
    sigma = sym.copy()
    for i in range(3):
        sigma[i,i] -= trace_third
    sigma2 = np.sum(sigma**2, axis=(0,1))
    return theta, sigma2, eigvals


def bootstrap_ci(delta, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(delta)
    boot_means = np.array([rng.choice(delta, size=n, replace=True).mean() for _ in range(n_boot)])
    return np.percentile(boot_means, [2.5, 97.5])


def QD_per_domain(theta, sigma2, grid_tree, observer_positions, radius_mpc):
    """Q_D within each spherical domain (baseline, no environment split)."""
    theta_flat = theta.ravel(); sigma2_flat = sigma2.ravel()
    qd_values = np.empty(len(observer_positions))
    for i, obs in enumerate(observer_positions):
        idx = grid_tree.query_ball_point(obs, r=radius_mpc)
        th = theta_flat[idx]; s2 = sigma2_flat[idx]
        var_theta = (th**2).mean() - th.mean()**2
        qd_values[i] = (2.0/3.0)*var_theta - s2.mean()
    return qd_values


## 2. Cosmic-web environment classification (V-web)

Each grid point is classified as Void, Sheet, Filament, or Knot by counting how many eigenvalues of the velocity-shear tensor exceed a threshold λ (Hoffman et al., 2012). The classification is computed on the IDW grid, not at tracer positions, which is essential for the classification to be volume-weighted rather than biased toward converging regions (see companion report, Section 2.2–2.3).

In [ ]:
def classify_web_grid(eigvals, threshold=0.0):
    """0 = Void, 1 = Sheet, 2 = Filament, 3 = Knot."""
    return np.sum(eigvals > threshold, axis=-1)


def largest_component_fraction_periodic(mask):
    """Fraction of True cells in the largest connected component, with
    periodic boundary conditions (required for a cosmological box)."""
    structure = ndimage.generate_binary_structure(3, 1)
    labels, n = ndimage.label(mask, structure=structure)
    if n == 0:
        return 0.0
    parent = list(range(n + 1))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for axis in range(3):
        face0 = np.take(labels, 0, axis=axis)
        face1 = np.take(labels, -1, axis=axis)
        both = (face0 > 0) & (face1 > 0)
        for a, b in zip(face0[both], face1[both]):
            union(a, b)
    roots = np.array([find(l) if l > 0 else 0 for l in labels.ravel()])
    sizes = np.bincount(roots[roots > 0])
    return sizes.max() / mask.sum() if mask.sum() > 0 else 0.0


def percolation_curve(eigvals, thresholds, min_eigs_above=2):
    """S(threshold): largest-connected-component fraction of the
    filament+knot class (>= min_eigs_above eigenvalues above threshold),
    as a function of threshold -- used to locate the percolation
    transition (Zel'dovich et al. 1982; Shandarin 1983)."""
    results = []
    for th in thresholds:
        labels = classify_web_grid(eigvals, threshold=th)
        mask = labels >= min_eigs_above
        frac_cells = mask.mean()
        S = largest_component_fraction_periodic(mask) if mask.sum() > 0 else 0.0
        results.append((th, frac_cells, S))
    return results


def void_percolation_curve(eigvals, thresholds):
    """Same as percolation_curve, for the Void class (0 eigenvalues above threshold)."""
    results = []
    for th in thresholds:
        mask = classify_web_grid(eigvals, threshold=th) == 0
        frac_cells = mask.mean()
        S = largest_component_fraction_periodic(mask) if mask.sum() > 0 else 0.0
        results.append((th, frac_cells, S))
    return results


## 3. Fiducial field at z = 1.0: reference computation and threshold calibration

The operating threshold λ is calibrated once, at z = 1.0, via percolation analysis: it is chosen at the point where the void network and the filament+knot network are simultaneously well connected (coexistence criterion, Forero-Romero et al., 2009). This reference field and threshold are reused as a fixed classification for all subsequent comparisons in this notebook.

In [ ]:
print("Downloading fiducial tracers, z=1.0...")
tracers_fid, velocities_fid, z_fid = load_tracers_and_velocities(
    flamingo_dir, VARIANT_A["run_path"], "halo_properties_0057.hdf5", expected_z=1.0)
print(f"  {len(tracers_fid)} tracers, z={z_fid:.3f}")

print("Building IDW velocity field...")
vx, vy, vz = build_velocity_field(tracers_fid, velocities_fid, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
theta_grid, sigma2_grid, eigvals_grid = compute_theta_sigma2_eigvals_grid(
    vx.astype(np.float64), vy.astype(np.float64), vz.astype(np.float64), CELL_SIZE)

print(f"Sanity check <theta> over the periodic box = {np.mean(theta_grid):.6f} (expected exactly 0)")


In [ ]:
th_lo, th_hi = np.percentile(eigvals_grid, [2, 98])
thresholds = np.linspace(th_lo, th_hi, 15)

curve_fk = percolation_curve(eigvals_grid, thresholds)
curve_void = void_percolation_curve(eigvals_grid, thresholds)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot([c[0] for c in curve_fk], [c[2] for c in curve_fk], "o-", label="Filament+Knot")
ax.plot([c[0] for c in curve_void], [c[2] for c in curve_void], "s-", label="Void")
ax.axvline(WEB_THRESHOLD, color="grey", linestyle="--", label=f"Operating threshold ({WEB_THRESHOLD})")
ax.set_xlabel("Threshold $\\lambda$ (velocity-shear eigenvalue)")
ax.set_ylabel("Largest connected component fraction $S$")
ax.set_title("Percolation curves, z=1.0 (fiducial)")
ax.legend()
plt.tight_layout()
plt.savefig("percolation_curve.png", dpi=150)
plt.show()

print(f"\nOperating threshold selected: lambda = {WEB_THRESHOLD} "
      f"(void and filament+knot networks both well connected, S~0.96)")


In [ ]:
labels_fiducial = classify_web_grid(eigvals_grid, threshold=WEB_THRESHOLD)

names = {0: "Void", 1: "Sheet", 2: "Filament", 3: "Knot"}
print("Volume fractions (fiducial, z=1.0):")
for i in range(4):
    print(f"  {names[i]:10s}: {np.mean(labels_fiducial==i)*100:5.2f}%")


## 4. Independent validation (T-web)

To verify that the V-web classification traces genuine cosmic structure rather than an artefact of using the velocity field, an independent classifier is built from the density field: the T-web (Hahn et al., 2007; Forero-Romero et al., 2009), based on the eigenvalues of the gravitational tidal tensor obtained by solving the Poisson equation in Fourier space. Agreement between the two, physically independent classifications is quantified with a confusion matrix.

In [ ]:
def cic_density(tracers, box_side, n_grid, weights=None):
    """Cloud-In-Cell density field from discrete tracer positions."""
    cell_size = box_side / n_grid
    pos_cell = tracers / cell_size
    i0 = np.floor(pos_cell).astype(int) % n_grid
    frac = pos_cell - np.floor(pos_cell)
    if weights is None:
        weights = np.ones(len(tracers))
    density = np.zeros((n_grid, n_grid, n_grid))
    for dx in [0,1]:
        for dy in [0,1]:
            for dz in [0,1]:
                w = (frac[:,0] if dx else (1-frac[:,0])) * \
                    (frac[:,1] if dy else (1-frac[:,1])) * \
                    (frac[:,2] if dz else (1-frac[:,2]))
                ix=(i0[:,0]+dx)%n_grid; iy=(i0[:,1]+dy)%n_grid; iz=(i0[:,2]+dz)%n_grid
                np.add.at(density, (ix,iy,iz), w*weights)
    return density


def tidal_tensor_eigvals(delta, box_side):
    """T-web (Hahn et al. 2007; Forero-Romero et al. 2009): Poisson equation
    in Fourier space, then eigenvalues of the tidal tensor. Positive
    eigenvalue = converging direction (verified against an isolated
    overdensity: all-positive eigenvalues at its centre)."""
    n = delta.shape[0]
    k1d = np.fft.fftfreq(n, d=box_side/n) * 2*np.pi
    kx, ky, kz = np.meshgrid(k1d, k1d, k1d, indexing='ij')
    k2 = kx**2+ky**2+kz**2; k2[0,0,0] = 1.0
    delta_k = np.fft.fftn(delta); delta_k[0,0,0] = 0.0
    T = np.zeros((n,n,n,3,3)); kvecs=[kx,ky,kz]
    for a in range(3):
        for b in range(a,3):
            Tij = np.real(np.fft.ifftn((kvecs[a]*kvecs[b]/k2)*delta_k))
            T[...,a,b]=Tij; T[...,b,a]=Tij
    return np.linalg.eigvalsh(T)


def confusion_matrix(labels_A, labels_B, name_A="V-web", name_B="T-web"):
    names = {0:"Void",1:"Sheet",2:"Filament",3:"Knot"}
    conf = np.zeros((4,4), dtype=int)
    for i in range(4):
        for j in range(4):
            conf[i,j] = np.sum((labels_A==i)&(labels_B==j))
    agreement = np.trace(conf)/conf.sum()
    print(f"Exact agreement {name_A} vs {name_B}: {agreement*100:.1f}% (chance level for 4 classes: 25%)")
    print(f"\nConfusion matrix (rows={name_A}, columns={name_B}, row %):")
    print(f"{'':10s}", *[f"{names[j]:>10s}" for j in range(4)])
    for i in range(4):
        frac_row = conf[i]/max(conf[i].sum(),1)
        print(f"{names[i]:10s}", *[f"{frac_row[j]*100:9.1f}%" for j in range(4)])
    return agreement, conf


In [ ]:
density_fid = cic_density(tracers_fid, BOX_SIDE, N_GRID)
delta_fid = (density_fid - density_fid.mean()) / density_fid.mean()
eigvals_T = tidal_tensor_eigvals(delta_fid, BOX_SIDE)

th_lo_T, th_hi_T = np.percentile(eigvals_T, [2, 98])
thresholds_T = np.linspace(th_lo_T, th_hi_T, 15)
curve_T_fk = percolation_curve(eigvals_T, thresholds_T)
curve_T_void = void_percolation_curve(eigvals_T, thresholds_T)

# coexistence threshold (same criterion as Section 3)
S_fk = np.array([c[2] for c in curve_T_fk])
S_void = np.array([c[2] for c in curve_T_void])
th_arr = np.array([c[0] for c in curve_T_fk])
idx_coexist = np.argmin(np.abs(S_fk - S_void))
T_WEB_THRESHOLD = th_arr[idx_coexist]
print(f"T-web operating threshold (coexistence criterion): {T_WEB_THRESHOLD:.3f}")

labels_T = classify_web_grid(eigvals_T, threshold=T_WEB_THRESHOLD)
agreement, conf = confusion_matrix(labels_fiducial, labels_T)


## 5. Exact variance decomposition of Q_D by environment

Because Q_D depends on the variance of θ, its split by environment is not a simple weighted average. The identity Q_D,total = Σᵢ fᵢ Q_D,ᵢ + Q_inter holds exactly (law of total variance), where Q_inter is the variance of the mean θ across environments. The identity is verified to machine precision below.

In [ ]:
def QD_by_environment(theta, sigma2, labels):
    names = {0: "Void", 1: "Sheet", 2: "Filament", 3: "Knot"}
    results = {}
    for env in range(4):
        mask = (labels == env)
        n = mask.sum()
        if n < 20:
            continue
        th = theta[mask]; s2 = sigma2[mask]
        mean_theta = th.mean()
        var_theta = (th**2).mean() - mean_theta**2
        QD = (2.0/3.0)*var_theta - s2.mean()
        results[names[env]] = {"N_cells": int(n), "volume_fraction": n/labels.size,
                                "mean_theta": mean_theta, "mean_sigma2": s2.mean(), "Q_D": QD}
    return results


def compute_Q_inter(theta, sigma2, labels):
    res = QD_by_environment(theta, sigma2, labels)
    means = np.array([d['mean_theta'] for d in res.values()])
    weights = np.array([d['N_cells']/labels.size for d in res.values()])
    global_mean = np.sum(means*weights)
    return (2.0/3.0) * np.sum(weights * (means-global_mean)**2)


res_env_fid = QD_by_environment(theta_grid, sigma2_grid, labels_fiducial)
Q_inter_fid = compute_Q_inter(theta_grid, sigma2_grid, labels_fiducial)
Q_D_total = (2.0/3.0)*np.var(theta_grid) - np.mean(sigma2_grid)
weighted_sum = sum(d['Q_D']*d['volume_fraction'] for d in res_env_fid.values())

print(f"Q_D total (fiducial, z=1.0)         = {Q_D_total:+.3f}")
print(f"sum_i f_i Q_i                        = {weighted_sum:+.3f}")
print(f"Q_inter                              = {Q_inter_fid:+.3f}")
print(f"sum_i f_i Q_i + Q_inter               = {weighted_sum + Q_inter_fid:+.3f}  "
      f"(identity check: matches Q_D total to {abs(Q_D_total-(weighted_sum+Q_inter_fid)):.2e})")


## 6. Differential test design: paired, tracer-count matched, per domain and environment

`QD_per_domain_by_environment` extends the baseline `QD_per_domain` (Section 1) by splitting each of the 80 independent spherical domains by cosmic-web environment before computing Q_D, preserving the validated bootstrap/Wilcoxon statistical protocol.

In [ ]:
def QD_per_domain_by_environment(theta, sigma2, labels, grid_tree, observer_positions,
                                  radius_mpc, min_cells=20):
    theta_flat = theta.ravel(); sigma2_flat = sigma2.ravel(); labels_flat = labels.ravel()
    names = {0: "Void", 1: "Sheet", 2: "Filament", 3: "Knot"}
    n_obs = len(observer_positions)
    results = {name: np.full(n_obs, np.nan) for name in names.values()}
    for i, obs in enumerate(observer_positions):
        idx = np.array(grid_tree.query_ball_point(obs, r=radius_mpc))
        th_dom = theta_flat[idx]; s2_dom = sigma2_flat[idx]; lab_dom = labels_flat[idx]
        for env_idx, env_name in names.items():
            mask = lab_dom == env_idx
            n = mask.sum()
            if n < min_cells:
                continue
            th = th_dom[mask]; s2 = s2_dom[mask]
            var_theta = (th**2).mean() - th.mean()**2
            results[env_name][i] = (2.0/3.0)*var_theta - s2.mean()
    return results


def paired_QD_by_environment(theta_A, sigma2_A, theta_B, sigma2_B, labels, grid_tree,
                              observer_positions, radius_mpc, min_cells=20):
    """Paired bootstrap + Wilcoxon comparison, per cosmic-web environment,
    over independent spherical domains -- same statistical protocol as the
    baseline (non-decomposed) global measurement."""
    res_A = QD_per_domain_by_environment(theta_A, sigma2_A, labels, grid_tree, observer_positions, radius_mpc, min_cells)
    res_B = QD_per_domain_by_environment(theta_B, sigma2_B, labels, grid_tree, observer_positions, radius_mpc, min_cells)
    output = {}
    for env in ["Void", "Sheet", "Filament", "Knot"]:
        qd_A = res_A[env]; qd_B = res_B[env]
        valid = ~np.isnan(qd_A) & ~np.isnan(qd_B)
        n_valid = valid.sum()
        if n_valid < 10:
            output[env] = {"n_domains": int(n_valid), "delta_mean": np.nan}
            continue
        delta = qd_B[valid] - qd_A[valid]
        ci_lo, ci_hi = bootstrap_ci(delta)
        try:
            _, p_wilcoxon = wilcoxon(qd_B[valid], qd_A[valid])
        except ValueError:
            p_wilcoxon = float("nan")
        output[env] = {"n_domains": int(n_valid), "delta_mean": delta.mean(),
                        "ci_lo": ci_lo, "ci_hi": ci_hi, "p_wilcoxon": p_wilcoxon}
    return output


## 7. Differential fiducial/NoCooling test by environment, three epochs

For each epoch, both variants are downloaded, tracer counts are matched (N-matching) for the differential test, and the paired per-environment comparison is run over the 80 independent domains. The full-resolution (non-matched) fiducial field is also used to track how the cosmic-web volume fractions evolve with epoch, applying the fixed reference threshold calibrated in Section 3. The cosmic-web classification (`labels_fiducial`) itself is kept fixed across epochs as the reference environment definition, consistent with the paired design used throughout.

In [ ]:
def process_epoch_differential(snapshot_file, z_expected, labels_ref):
    print(f"\n{'='*70}\nEpoch z={z_expected}\n{'='*70}")
    tr_f, vel_f, _ = load_tracers_and_velocities(flamingo_dir, VARIANT_A["run_path"], snapshot_file, z_expected)
    tr_n, vel_n, _ = load_tracers_and_velocities(flamingo_dir, VARIANT_B["run_path"], snapshot_file, z_expected)
    print(f"  tracers: fiducial={len(tr_f)}, NoCooling={len(tr_n)}")

    # full-resolution fiducial field: used to track cosmic-web volume
    # fraction evolution with the fixed reference threshold
    vx_full, vy_full, vz_full = build_velocity_field(tr_f, vel_f, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
    _, _, eigvals_full = compute_theta_sigma2_eigvals_grid(
        vx_full.astype(np.float64), vy_full.astype(np.float64), vz_full.astype(np.float64), CELL_SIZE)
    labels_full = classify_web_grid(eigvals_full, threshold=WEB_THRESHOLD)
    volume_fractions = {names[i]: np.mean(labels_full == i) for i in range(4)}

    # N-matched fields: used for the paired differential test
    target_n = min(len(tr_f), len(tr_n))
    tr_f_m, vel_f_m = subsample_to_match(tr_f, vel_f, target_n, seed=SEED)
    tr_n_m, vel_n_m = subsample_to_match(tr_n, vel_n, target_n, seed=SEED)

    vx_f, vy_f, vz_f = build_velocity_field(tr_f_m, vel_f_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
    theta_f, sigma2_f, _ = compute_theta_sigma2_eigvals_grid(
        vx_f.astype(np.float64), vy_f.astype(np.float64), vz_f.astype(np.float64), CELL_SIZE)
    print(f"  <theta> fiducial (N-matched)  = {np.mean(theta_f):.6f} (expected 0)")

    vx_n, vy_n, vz_n = build_velocity_field(tr_n_m, vel_n_m, GRID_1D, BOX_SIDE, N_GRID, K_NEIGHBORS)
    theta_n, sigma2_n, _ = compute_theta_sigma2_eigvals_grid(
        vx_n.astype(np.float64), vy_n.astype(np.float64), vz_n.astype(np.float64), CELL_SIZE)
    print(f"  <theta> NoCooling (N-matched) = {np.mean(theta_n):.6f} (expected 0)")

    # exact variance decomposition, this epoch (fiducial, full resolution)
    Q_D_tot = (2.0/3.0)*np.var(theta_f) - np.mean(sigma2_f)
    res_env = QD_by_environment(theta_f, sigma2_f, labels_ref)
    Q_inter = compute_Q_inter(theta_f, sigma2_f, labels_ref)

    result = paired_QD_by_environment(theta_f, sigma2_f, theta_n, sigma2_n, labels_ref,
                                       grid_tree, observer_positions, DOMAIN_RADIUS_MPC)
    print(f"\n  {'Environment':10s} {'Delta':>10s} {'95% CI':>22s} {'p':>10s}")
    for env, d in result.items():
        if np.isnan(d.get('delta_mean', np.nan)):
            print(f"  {env:10s}: insufficient data"); continue
        print(f"  {env:10s} {d['delta_mean']:+10.4f}  [{d['ci_lo']:+7.3f},{d['ci_hi']:+7.3f}]  {d['p_wilcoxon']:10.2e}")

    return {"differential": result, "volume_fractions": volume_fractions,
            "Q_D_total": Q_D_tot, "Q_by_env": res_env, "Q_inter": Q_inter}


names = {0: "Void", 1: "Sheet", 2: "Filament", 3: "Knot"}
epoch_results = {}
for snapshot_file, z in EPOCHS:
    epoch_results[z] = process_epoch_differential(snapshot_file, z, labels_fiducial)


## 8. Summary plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# (a) volume fractions vs z
ax = axes[0, 0]
z_arr = sorted(epoch_results.keys(), reverse=True)
for env in ["Void", "Sheet", "Filament", "Knot"]:
    vals = [epoch_results[z]["volume_fractions"][env]*100 for z in z_arr]
    ax.plot(z_arr, vals, "o-", label=env)
ax.set_xlabel("z"); ax.set_ylabel("Volume fraction (%)")
ax.set_title("Cosmic-web volume fractions vs epoch (fiducial)")
ax.invert_xaxis(); ax.legend()

# (b) Q_D decomposition vs z (stacked components + total)
ax = axes[0, 1]
q_tot = [epoch_results[z]["Q_D_total"] for z in z_arr]
q_inter = [epoch_results[z]["Q_inter"] for z in z_arr]
ax.plot(z_arr, q_tot, "ko-", label="Q_D total")
ax.plot(z_arr, q_inter, "s--", color="tab:red", label="Q_inter")
ax.set_xlabel("z"); ax.set_ylabel("Q_D  [(km/s/Mpc)$^2$]")
ax.set_title("Q_D total and inter-environment term vs epoch")
ax.invert_xaxis(); ax.legend()
ax.axhline(0, color="grey", linewidth=0.8)

# (c) differential Delta vs z, by environment, with 95% CI
ax = axes[1, 0]
colors = {"Void": "tab:blue", "Sheet": "tab:orange", "Filament": "tab:green", "Knot": "tab:red"}
for env in ["Void", "Sheet", "Filament", "Knot"]:
    zs, deltas, lo, hi = [], [], [], []
    for z in z_arr:
        d = epoch_results[z]["differential"][env]
        if np.isnan(d.get("delta_mean", np.nan)):
            continue
        zs.append(z); deltas.append(d["delta_mean"]); lo.append(d["delta_mean"]-d["ci_lo"]); hi.append(d["ci_hi"]-d["delta_mean"])
    ax.errorbar(zs, deltas, yerr=[lo, hi], fmt="o-", capsize=4, label=env, color=colors[env])
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("z"); ax.set_ylabel("$\\Delta Q_D$ = NoCooling $-$ fiducial  [(km/s/Mpc)$^2$]")
ax.set_title("Differential Q_D by environment vs epoch (N-matched, 95% CI)")
ax.invert_xaxis(); ax.legend()

# (d) V-web vs T-web confusion matrix (z=1.0)
ax = axes[1, 1]
conf_frac = conf / conf.sum(axis=1, keepdims=True)
im = ax.imshow(conf_frac, cmap="Blues", vmin=0, vmax=1)
labels_names = ["Void", "Sheet", "Filament", "Knot"]
ax.set_xticks(range(4)); ax.set_xticklabels(labels_names)
ax.set_yticks(range(4)); ax.set_yticklabels(labels_names)
ax.set_xlabel("T-web"); ax.set_ylabel("V-web")
ax.set_title(f"V-web vs T-web confusion matrix, z=1.0\n(agreement={agreement*100:.1f}%, chance=25%)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{conf_frac[i,j]*100:.0f}%", ha="center", va="center",
                color="white" if conf_frac[i,j]>0.5 else "black", fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig("summary_plots.png", dpi=150)
plt.show()


## 9. Summary table

In [ ]:
print(f"{'z':>5s} {'Environment':12s} {'Delta':>10s} {'95% CI':>22s} {'p':>10s}")
for z in z_arr:
    for env in ["Void", "Sheet", "Filament", "Knot"]:
        d = epoch_results[z]["differential"][env]
        if np.isnan(d.get("delta_mean", np.nan)):
            print(f"{z:5.1f} {env:12s} {'insufficient data':>44s}")
            continue
        delta_mean = d["delta_mean"]
        ci_lo = d["ci_lo"]
        ci_hi = d["ci_hi"]
        p_val = d["p_wilcoxon"]
        print(f"{z:5.1f} {env:12s} {delta_mean:+10.4f}  [{ci_lo:+7.3f},{ci_hi:+7.3f}]  {p_val:10.2e}")
